# Watermark Comparison: WavMark vs AudioSeal
**Project: Persian watermarking on speech synthesized with Mana-Persian-Piper**

The same 10 Persian sentences, the same 16-bit message, the same 11 attacks as stage 2 — only this time the watermark is put on with **WavMark** (Chen et al., 2023), not AudioSeal. At the end of the notebook we build a head-to-head table and see which method wins where.

**The hypothesis we are testing:** speed (0.9x and 1.1x) will probably break WavMark more than AudioSeal — because WavMark places the message in 1-second chunks with a sync pattern, and speeding up or slowing down the audio scrambles that pattern; while for AudioSeal, speed only dropped the probability but the bits stayed 95–97% intact. This is only a hypothesis — the numbers decide, not us.

**Three important differences between WavMark and AudioSeal that affect the comparison:**
- **Capacity:** WavMark puts 32 bits in every 1 second (16 bits sync pattern + 16 bits payload) and repeats it across the file; AudioSeal puts only 16 bits in the whole file.
- **Detection:** WavMark has no "probability" — either it finds the sync pattern and returns the message, or it does not. So our comparison column is "how many of the 10 sentences were detected" and "bit accuracy", not probability.
- **Minimum length:** WavMark does not work below 1 second. Our sentences are about 2 to 4 seconds, so there is no problem — but cell 7 checks the length of each sentence.


## Cell 1 — Environment check: do we have a GPU?
Like stage 2: if there is no GPU, select `Runtime > Change runtime type > T4 GPU` from the top menu.


In [ ]:
import sys, torch
print("Python:", sys.version)
print("torch:", torch.__version__)
print("Is CUDA available?", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Cell 2 — Install
`wavmark` version 0.0.2 is installed from PyPI (its model is downloaded on first use, so internet is needed). AudioSeal is not installed here — we take the AudioSeal numbers from the final stage-2 table, not by running it again.


In [ ]:
!pip install -q persian-speech wavmark soundfile faster-whisper scipy 2>&1 | tail -3
print("Install done")


## Cell 3 — Check the install


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
try:
    torch._dynamo.config.suppress_errors = True
except Exception as e:
    print("dynamo setting skipped:", e)
import persian_speech_runtime, wavmark, faster_whisper, scipy
print("All packages installed")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


## Cell 4 — The same 10 fixed Persian sentences + the same 16-bit message
Do not change the sentences or the message — only then are this notebook's numbers comparable with stages 1b and 2.


In [ ]:
# The 10 fixed Persian test sentences — kept in Persian in every stage so the numbers stay comparable
SENTENCES = [
    "سلام، حالت چطور است؟",
    "هوا امروز در تهران آفتابی است.",
    "من هر روز صبح کتاب می‌خوانم.",
    "هوش مصنوعی دارد زندگی ما را تغییر می‌دهد.",
    "این آزمایش برای بررسی واترمارک فارسی انجام می‌شود.",
    "لطفا در را ببند و چراغ را خاموش کن.",
    "دانشجویان در آزمایشگاه روی پروژه گفتار کار می‌کنند.",
    "صدای ساخته‌شده باید طبیعی و قابل فهم باشد.",
    "ما می‌خواهیم بفهمیم واترمارک روی فارسی هم می‌ماند یا نه.",
    "فردا صبح ساعت نه به دانشگاه می‌روم.",
]
for i, s in enumerate(SENTENCES, 1):
    print(f"{i:02d}. {s}  ({len(s.split())} words)")
print()
print("Sentence count:", len(SENTENCES))
print("Total words:", sum(len(s.split()) for s in SENTENCES))

# The same 16-bit message as stages 1, 1b and 2 — WavMark takes these 16 bits as its payload
# and adds its own 16-bit sync pattern (32 bits per second in total).
import numpy as np
PAYLOAD = np.array([1,0,1,1,0,1,0,0,1,1,0,1,0,1,1,0], dtype=int)
print("Fixed 16-bit message:", PAYLOAD.tolist())


## Cell 5 — Load Mana-Persian-Piper
Exactly like stages 1b and 2. Its output is 22050 Hz and is converted to 16000 in the next cell.


In [ ]:
from persian_speech_runtime import PiperSpeechEngine
tts = PiperSpeechEngine()
print("Mana-Persian-Piper ready.")
print("Note: this model's output is 22050 Hz; the next cell converts it to 16000.")


## Cell 6 — Single-sentence test: synthesize only sentence 1
If this one is broken, do not build the rest.


In [ ]:
import os, subprocess, soundfile as sf
from IPython.display import Audio, display

os.makedirs("persian_tts_clean", exist_ok=True)
os.makedirs("tmp_22050", exist_ok=True)

def synth_mana_16k(text, out_16k):
    tmp = out_16k.replace("persian_tts_clean", "tmp_22050").replace("_clean.wav", "_22050.wav")
    tts.save(text, tmp)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp,"-ar","16000",out_16k], check=True)
    wav, sr = sf.read(out_16k)
    assert sr == 16000, f"Sample rate is {sr}, must be 16000"
    return wav, sr

wav1, sr = synth_mana_16k(SENTENCES[0], "persian_tts_clean/utt_001_clean.wav")
print("Sentence 1:", SENTENCES[0])
print("Audio length:", round(len(wav1)/sr, 2), "seconds | samples:", len(wav1), "| rate:", sr)
display(Audio(wav1, rate=sr))
print("Saved: persian_tts_clean/utt_001_clean.wav")


## Cell 7 — Synthesize all 10 clean sentences (16 kHz) + minimum-length check
WavMark puts its message in every 1-second chunk, so each sentence must be at least 1 second long. If a sentence is shorter than 1 second, stop and tell me.


In [ ]:
import numpy as np, soundfile as sf, time
durations = []
t0 = time.time()
for i, text in enumerate(SENTENCES, 1):
    wav, sr = synth_mana_16k(text, f"persian_tts_clean/utt_{i:03d}_clean.wav")
    d = len(wav)/sr
    durations.append(d)
    print(f"{i:02d}. {d:.2f}s  persian_tts_clean/utt_{i:03d}_clean.wav")
print()
print(f"Total audio duration: {sum(durations):.2f} seconds")
print(f"Shortest sentence: {min(durations):.2f} seconds — must be over 1 second")
assert min(durations) > 1.0, "A sentence is shorter than 1 second; WavMark does not work on it"
print(f"Synthesis time (wall-clock): {time.time()-t0:.1f} seconds")


## Cell 8 — Load WavMark + helper functions
- Model: `wavmark.load_model()` — like AudioSeal, it goes to the GPU if there is one, otherwise CPU.
- Encode: `wavmark.encode_watermark(model, signal, payload)` — repeats the same 16-bit message in every 1 second.
- Decode: `wavmark.decode_watermark(model, signal)` — if it finds the sync pattern it returns the 16-bit message; if not, it returns `None` (or raises). So "detected" means decode returned a 16-bit message — not a probability.
- We also compute the **watermark SNR** to compare how inaudible WavMark is against AudioSeal (in stage 1b, SNR for sentences 1, 5 and 10 was 32.8, 34.3 and 27.9 dB).


In [ ]:
import torch
import wavmark
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading WavMark...")
model = wavmark.load_model().to(DEVICE)
model.eval()
print("WavMark ready. Device:", DEVICE)

def embed_watermark(wav_np):
    sig = np.asarray(wav_np, dtype=np.float32)
    if sig.ndim > 1:
        sig = sig[:, 0]
    wm_signal, _ = wavmark.encode_watermark(model, sig, PAYLOAD, show_progress=False)
    return np.asarray(wm_signal, dtype=np.float32)

def decode_payload(wav_np):
    """Returns: (detected?, decoded bits or None).
    If the sync pattern is not found, decode returns None or raises —
    both mean "not detected", not that the notebook is broken."""
    sig = np.asarray(wav_np, dtype=np.float32)
    if sig.ndim > 1:
        sig = sig[:, 0]
    try:
        decoded, _ = wavmark.decode_watermark(model, sig, show_progress=False)
    except Exception as e:
        print("  (decode raised an error, counted as not detected:", type(e).__name__, ")")
        return False, None
    if decoded is None:
        return False, None
    bits = np.asarray(decoded).reshape(-1).astype(int).tolist()
    if len(bits) != 16:
        return False, None
    return True, bits

def bit_accuracy(decoded):
    true = PAYLOAD.astype(int).tolist()
    correct = sum(int(a) == int(b) for a, b in zip(decoded, true))
    return correct, correct / len(true)

def snr_db(clean, wm):
    clean = np.asarray(clean, dtype=np.float64)
    diff = clean - np.asarray(wm, dtype=np.float64)
    return 10 * np.log10(np.mean(clean**2) / (np.mean(diff**2) + 1e-12))


## Cell 9 — WavMark watermark for all 10 sentences + baseline
The baseline should be 10 out of 10 — if it is not, stop and tell me. We also try decoding the clean audio: it should give "not detected" (if audio without a watermark is detected as watermarked, we have false positives).


In [ ]:
import soundfile as sf
os.makedirs("persian_tts_watermarked_wavmark", exist_ok=True)
baseline = []
for i, text in enumerate(SENTENCES, 1):
    wav_clean, sr = sf.read(f"persian_tts_clean/utt_{i:03d}_clean.wav")
    det_clean, _ = decode_payload(wav_clean)
    wav_wm = embed_watermark(wav_clean)
    sf.write(f"persian_tts_watermarked_wavmark/utt_{i:03d}_wm.wav", wav_wm, sr)
    det_wm, bits_wm = decode_payload(wav_wm)
    correct, acc = bit_accuracy(bits_wm) if det_wm else (0, 0.0)
    s = snr_db(wav_clean, wav_wm)
    baseline.append({"id": i, "clean_detected": bool(det_clean), "wm_detected": bool(det_wm),
                     "bits_correct": correct, "snr_db": round(float(s), 1)})
    print(f"{i:02d}. clean={'DETECTED!' if det_clean else 'no'}  watermarked={'yes' if det_wm else 'no'}  bits={correct}/16  SNR={s:.1f} dB")
print()
print("Baseline (no attack) ready — should be 10/10, and clean audio must not be detected.")
print(f"Mean watermark SNR: {np.mean([b['snr_db'] for b in baseline]):.1f} dB")


## Cell 10 — Define the attacks
**Exactly the same 11 attacks as stage 2, with the same parameters** — only then is the comparison fair:
- **MP3:** encoded and decoded with ffmpeg (bitrate is a parameter).
- **Noise:** Gaussian white noise is added; its strength is set by SNR (20 dB = little, 10 dB = a lot). A fixed seed keeps it reproducible.
- **Speed:** ffmpeg's atempo filter; 0.9 means slower and longer, 1.1 means faster and shorter.
- **Gain down:** multiply by 0.5 (minus 6 dB).
- **Lowpass filter:** 4th-order Butterworth with a 4 kHz cutoff.
- **8 kHz resampling:** round trip 16 → 8 → 16 kHz (telephone quality).


In [ ]:
import subprocess, numpy as np, soundfile as sf
from scipy.signal import butter, sosfiltfilt

def attack_mp3(wav_np, sr, bitrate_kbps, tmp_in="/tmp/atk_in.wav", tmp_mp3="/tmp/atk.mp3", tmp_out="/tmp/atk_out.wav"):
    sf.write(tmp_in, wav_np, sr)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_in,"-codec:a","libmp3lame","-b:a",f"{bitrate_kbps}k",tmp_mp3], check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_mp3,"-ar","16000",tmp_out], check=True)
    wav, sr2 = sf.read(tmp_out)
    assert sr2 == 16000
    return wav.astype(np.float32)

def attack_noise(wav_np, sr, snr_db, seed):
    rng = np.random.default_rng(seed)
    sig_rms = float(np.sqrt(np.mean(wav_np**2))) + 1e-12
    noise_rms = sig_rms / (10 ** (snr_db / 20))
    noise = rng.normal(0, noise_rms, size=wav_np.shape).astype(np.float32)
    return np.clip(wav_np + noise, -1.0, 1.0).astype(np.float32)

def attack_speed(wav_np, sr, factor, tmp_in="/tmp/atk_in.wav", tmp_out="/tmp/atk_out.wav"):
    sf.write(tmp_in, wav_np, sr)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_in,"-filter:a",f"atempo={factor}","-ar","16000",tmp_out], check=True)
    wav, sr2 = sf.read(tmp_out)
    assert sr2 == 16000
    return wav.astype(np.float32)

def attack_gain(wav_np, sr, factor):
    return np.clip(wav_np * factor, -1.0, 1.0).astype(np.float32)

def attack_lowpass(wav_np, sr, cutoff_hz):
    sos = butter(4, cutoff_hz, btype="low", fs=sr, output="sos")
    return sosfiltfilt(sos, wav_np).astype(np.float32)

def attack_resample8k(wav_np, sr, tmp_in="/tmp/atk_in.wav", tmp_mid="/tmp/atk_8k.wav", tmp_out="/tmp/atk_out.wav"):
    sf.write(tmp_in, wav_np, sr)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_in,"-ar","8000",tmp_mid], check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_mid,"-ar","16000",tmp_out], check=True)
    wav, sr2 = sf.read(tmp_out)
    assert sr2 == 16000
    return wav.astype(np.float32)

ATTACKS = [
    ("mp3_256",    "MP3 256k",            lambda w, sr, i: attack_mp3(w, sr, 256)),
    ("mp3_128",    "MP3 128k",            lambda w, sr, i: attack_mp3(w, sr, 128)),
    ("mp3_64",     "MP3 64k",             lambda w, sr, i: attack_mp3(w, sr, 64)),
    ("mp3_32",     "MP3 32k",             lambda w, sr, i: attack_mp3(w, sr, 32)),
    ("noise_20db", "Noise SNR 20dB",      lambda w, sr, i: attack_noise(w, sr, 20, seed=1000+i)),
    ("noise_10db", "Noise SNR 10dB",      lambda w, sr, i: attack_noise(w, sr, 10, seed=1000+i)),
    ("speed_0.9",  "Speed 0.9x",          lambda w, sr, i: attack_speed(w, sr, 0.9)),
    ("speed_1.1",  "Speed 1.1x",          lambda w, sr, i: attack_speed(w, sr, 1.1)),
    ("gain_0.5",   "Gain 0.5 (-6dB)",     lambda w, sr, i: attack_gain(w, sr, 0.5)),
    ("lowpass_4k", "Lowpass 4kHz",        lambda w, sr, i: attack_lowpass(w, sr, 4000)),
    ("resample_8k","Resample 8kHz",       lambda w, sr, i: attack_resample8k(w, sr)),
]
print(f"Attack count: {len(ATTACKS)}")
for name, label, _ in ATTACKS:
    print(f" - {name}: {label}")


## Cell 11 — Apply all attacks to all 10 sentences
Output: `persian_wm_attacked_wavmark/<attack>/utt_001_wm.wav` to `utt_010_wm.wav` — that is 11 attacks × 10 sentences = 110 files.


In [ ]:
import os, soundfile as sf, time
os.makedirs("persian_wm_attacked_wavmark", exist_ok=True)
t0 = time.time()
for name, label, fn in ATTACKS:
    os.makedirs(f"persian_wm_attacked_wavmark/{name}", exist_ok=True)
    for i in range(1, 11):
        wav_wm, sr = sf.read(f"persian_tts_watermarked_wavmark/utt_{i:03d}_wm.wav")
        wav_atk = fn(wav_wm, sr, i)
        sf.write(f"persian_wm_attacked_wavmark/{name}/utt_{i:03d}_wm.wav", wav_atk, 16000)
    print(f"{name} ({label}): 10 files created")
print()
print(f"Total attacked files: {len(ATTACKS)*10} files in {time.time()-t0:.1f} seconds")


## Cell 12 — WavMark detection on all attacked files
Per attack: how many of the 10 sentences were detected, mean bit accuracy.
**Note:** when WavMark decode cannot find the pattern, it scans the whole file with a sliding window — so this cell, especially on broken attacks, may take a few minutes. That slowness is normal, not a failure.


In [ ]:
import soundfile as sf
rows = []
for name, label, _ in ATTACKS:
    accs = []
    detected_n = 0
    for i in range(1, 11):
        wav_atk, sr = sf.read(f"persian_wm_attacked_wavmark/{name}/utt_{i:03d}_wm.wav")
        det, bits = decode_payload(wav_atk)
        correct, acc = bit_accuracy(bits) if det else (0, 0.0)
        rows.append({"id": i, "attack": name, "attack_label": label, "detected": bool(det),
                     "bits_correct": correct, "bit_accuracy": round(acc, 3)})
        detected_n += int(det)
        accs.append(acc)
    print(f"{name:12s} detected={detected_n}/10  mean bit acc={np.mean(accs):.2%}")
print()
print("Detection on all attacks done.")


## Cell 13 — Listen: sentence 1 after the attacks
Before the quality numbers, listen yourself: how badly do MP3 at 32k and noise at 10dB damage the audio? Can you hear the WavMark watermark?


In [ ]:
from IPython.display import Audio, display
import soundfile as sf
print("— Sentence 1, watermarked, no attack (WavMark) —")
wav, sr = sf.read("persian_tts_watermarked_wavmark/utt_001_wm.wav"); display(Audio(wav, rate=sr))
print("— Sentence 1, after MP3 32k —")
wav, sr = sf.read("persian_wm_attacked_wavmark/mp3_32/utt_001_wm.wav"); display(Audio(wav, rate=sr))
print("— Sentence 1, after noise 10dB —")
wav, sr = sf.read("persian_wm_attacked_wavmark/noise_10db/utt_001_wm.wav"); display(Audio(wav, rate=sr))
print("— Sentence 1, after speed 1.1x —")
wav, sr = sf.read("persian_wm_attacked_wavmark/speed_1.1/utt_001_wm.wav"); display(Audio(wav, rate=sr))


## Cell 14 — Quality: WER with Persian Whisper small
For every condition (no attack + 11 attacks) we transcribe all 10 sentences with Whisper small and compare with the original sentence.
WER means "word error rate": what percentage of words were misheard. Low WER = the speech is still intelligible.
To keep the comparison fair, we convert the half-space to a space and turn Arabic «ي/ك» back into Persian «ی/ک» — in the reference text and in Whisper's output alike.

**Important note:** we run Whisper on CPU on purpose, because faster-whisper on Colab GPU errors with `libcublas.so.12` missing. The small model is small and fast enough on CPU.

**Why small and not tiny?** In stage 2 with tiny, WER was about 95–100% everywhere — even 93% with no attack — because tiny does not understand Persian well. So the WER column with tiny is useless for comparing attacks; small is more accurate and shows the real range.


In [ ]:
from faster_whisper import WhisperModel
# Note: we run Whisper on CPU on purpose.
# faster-whisper on Colab GPU cannot find the libcublas.so.12 library and errors out.
# The small model is fast enough on CPU; WavMark stays on GPU.
WHISPER_DEVICE = "cpu"
print("Loading Whisper small on CPU ...")
whisper = WhisperModel("small", device=WHISPER_DEVICE, compute_type="int8")
print("Whisper small ready.")

def normalize_fa(t):
    t = t.replace("ي", "ی").replace("ك", "ک").replace("\u200c", " ")
    for ch in "؟،.,!؛:()«»\"'":
        t = t.replace(ch, " ")
    return t.split()

def wer(ref, hyp):
    r, h = normalize_fa(ref), normalize_fa(hyp)
    n, m = len(r), len(h)
    if n == 0:
        return 1.0 if m else 0.0
    d = list(range(m + 1))
    for i in range(1, n + 1):
        prev, d[0] = d[0], i
        for j in range(1, m + 1):
            cur = d[j]
            d[j] = min(d[j] + 1, d[j - 1] + 1, prev + (0 if r[i - 1] == h[j - 1] else 1))
            prev = cur
    return d[m] / n

def transcribe_wer(wav_np, ref_text):
    segments, _ = whisper.transcribe(wav_np.astype(np.float32), language="fa", beam_size=1)
    hyp = "".join(s.text for s in segments).strip()
    return wer(ref_text, hyp), hyp

print("WER test on sentence 1, no attack:")
wav, sr = sf.read("persian_tts_watermarked_wavmark/utt_001_wm.wav")
w, hyp = transcribe_wer(wav, SENTENCES[0])
print("Reference:    ", SENTENCES[0])
print("Heard:", hyp)
print(f"WER, sentence 1: {w:.2%}")


## Cell 15 — Compute WER for all conditions
12 conditions (no attack + 11 attacks) × 10 sentences = 120 transcriptions. On CPU it takes about 15 to 30 minutes (the small model is bigger and more accurate than tiny).


In [ ]:
import soundfile as sf, time
wer_rows = []
t0 = time.time()
conditions = [("clean_wm", "No attack", "persian_tts_watermarked_wavmark")] + [(name, label, f"persian_wm_attacked_wavmark/{name}") for name, label, _ in ATTACKS]
for cond_name, cond_label, folder in conditions:
    wers = []
    for i in range(1, 11):
        wav, sr = sf.read(f"{folder}/utt_{i:03d}_wm.wav")
        w, hyp = transcribe_wer(wav, SENTENCES[i - 1])
        wer_rows.append({"id": i, "attack": cond_name, "wer": round(w, 3)})
        wers.append(w)
    print(f"{cond_name:12s} mean WER={np.mean(wers):.2%}")
print()
print(f"All WERs computed in {time.time()-t0:.1f} seconds.")


## Cell 16 — WavMark summary table
Detection + WER go into one file: `summary_wavmark.csv` — each row is one sentence under one attack.


In [ ]:
import pandas as pd
det_df = pd.DataFrame(rows)
wer_df = pd.DataFrame(wer_rows)
summary = det_df.merge(wer_df, on=["id", "attack"], how="left")
base_rows = []
for b in baseline:
    w = wer_df[(wer_df["id"] == b["id"]) & (wer_df["attack"] == "clean_wm")]["wer"].values
    base_rows.append({"id": b["id"], "attack": "clean_wm", "attack_label": "No attack",
                      "detected": bool(b["wm_detected"]), "bits_correct": b["bits_correct"],
                      "bit_accuracy": round(b["bits_correct"] / 16, 3),
                      "wer": float(w[0]) if len(w) else None, "snr_db": b["snr_db"]})
summary = pd.concat([pd.DataFrame(base_rows), summary], ignore_index=True)
summary.to_csv("summary_wavmark.csv", index=False, encoding="utf-8")
print("Saved: summary_wavmark.csv")
pivot = summary.groupby(["attack", "attack_label"]).agg(detected=("detected", "sum"),
    mean_bits=("bit_accuracy", "mean"), mean_wer=("wer", "mean")).reset_index()
print(pivot.to_string(index=False))


## Cell 17 — WavMark charts
Four panels (labels are in English so the matplotlib font does not break):
- Top-left: number of detections out of 10 per attack (WavMark has no probability, so we plot the detection rate)
- Top-right: mean bit accuracy per attack
- Bottom-left: mean WER with Whisper **small** per attack (audio quality)
- Bottom-right: MP3 knee curve — detection rate vs bitrate


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, numpy as np
summary = pd.read_csv("summary_wavmark.csv")
pivot = summary.groupby(["attack", "attack_label"]).agg(detected=("detected", "sum"),
    mean_bits=("bit_accuracy", "mean"), mean_wer=("wer", "mean")).reset_index()
order = ["clean_wm"] + [a[0] for a in ATTACKS]
pivot["ord"] = pivot["attack"].apply(lambda a: order.index(a))
pivot = pivot.sort_values("ord")
labels = pivot["attack"].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
x = np.arange(len(pivot))
axes[0, 0].bar(x, pivot["detected"]); axes[0, 0].axhline(5, linestyle="--")
axes[0, 0].set_xticks(x); axes[0, 0].set_xticklabels(labels, rotation=45, ha="right")
axes[0, 0].set_ylabel("Detected (out of 10)"); axes[0, 0].set_title("WavMark detection per attack")
axes[0, 0].set_ylim(0, 10.5); axes[0, 0].grid(axis="y", alpha=0.3)
axes[0, 1].bar(x, pivot["mean_bits"])
axes[0, 1].set_xticks(x); axes[0, 1].set_xticklabels(labels, rotation=45, ha="right")
axes[0, 1].set_ylabel("Mean bit accuracy"); axes[0, 1].set_title("Bit accuracy per attack (of 16 bits)")
axes[0, 1].set_ylim(0, 1.05); axes[0, 1].grid(axis="y", alpha=0.3)
axes[1, 0].bar(x, pivot["mean_wer"])
axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(labels, rotation=45, ha="right")
axes[1, 0].set_ylabel("Mean WER"); axes[1, 0].set_title("Whisper small WER per attack (quality)")
axes[1, 0].set_ylim(0, 1.05); axes[1, 0].grid(axis="y", alpha=0.3)
mp3 = pivot[pivot["attack"].str.startswith("mp3")].copy()
mp3["kbps"] = mp3["attack"].str.replace("mp3_", "").astype(int)
mp3 = mp3.sort_values("kbps")
axes[1, 1].plot(mp3["kbps"], mp3["detected"], marker="o")
axes[1, 1].axhline(5, linestyle="--")
axes[1, 1].set_xscale("log"); axes[1, 1].set_xticks(mp3["kbps"]); axes[1, 1].get_xaxis().set_major_formatter(plt.ScalarFormatter())
axes[1, 1].set_xlabel("MP3 bitrate (kbps)"); axes[1, 1].set_ylabel("Detected (out of 10)")
axes[1, 1].set_title("MP3 knee: detection vs bitrate"); axes[1, 1].set_ylim(0, 10.5); axes[1, 1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("chart_wavmark.png", dpi=150); plt.show()
print("Saved: chart_wavmark.png")


## Cell 18 — Where is the knee?
Our definition for WavMark: an attack is "broken" if fewer than 5 of 10 sentences are detected or bit accuracy drops below 90%.
This cell prints the first breaking point for each attack family.


In [ ]:
print("=== Breaking point (knee) — WavMark ===")
print("Definition: detection below 5 of 10, or bit accuracy below 90%, means broken.")
print()
for fam, names in [("MP3", ["mp3_256", "mp3_128", "mp3_64", "mp3_32"]), ("Noise", ["noise_20db", "noise_10db"]), ("Speed", ["speed_0.9", "speed_1.1"])]:
    print(f"— {fam}:")
    for n in names:
        r = pivot[pivot["attack"] == n].iloc[0]
        status = "intact" if r["detected"] >= 5 and r["mean_bits"] >= 0.9 else "broken"
        print(f"   {n:12s} detected={int(r['detected'])}/10  bit acc={r['mean_bits']:.2%}  WER={r['mean_wer']:.2%}  {status}")
print()
print("— Single attacks:")
for n in ["gain_0.5", "lowpass_4k", "resample_8k"]:
    r = pivot[pivot["attack"] == n].iloc[0]
    status = "intact" if r["detected"] >= 5 and r["mean_bits"] >= 0.9 else "broken"
    print(f"   {n:12s} detected={int(r['detected'])}/10  bit acc={r['mean_bits']:.2%}  WER={r['mean_wer']:.2%}  {status}")


## Cell 19 — Head-to-head table: AudioSeal vs WavMark
The AudioSeal numbers come from the **fresh stage-2 run with Whisper small** (13 November 2026, 735.9 seconds) — the same table you just sent and that was verified. Each method's WER was measured in its own run, so compare the WER column with care: the fresh stage-2 run gave a clean WER of 80.44%, while its earlier run gave 77.66% — that shift is run-to-run variation, not a watermark effect.


In [ ]:
# AudioSeal — fresh stage-2 run (Whisper small), verified 13 November 2026
AUDIOSEAL_REF = {
    "clean_wm":    (10, 1.000, 0.8044),
    "mp3_256":     (10, 1.000, 0.8600),
    "mp3_128":     (10, 1.000, 0.8461),
    "mp3_64":      (10, 1.000, 0.8822),
    "mp3_32":      (10, 1.000, 0.8461),
    "noise_20db":  ( 1, 0.850, 0.8349),
    "noise_10db":  ( 0, 0.612, 1.0030),
    "speed_0.9":   ( 0, 0.950, 0.8030),
    "speed_1.1":   ( 0, 0.969, 0.8710),
    "gain_0.5":    (10, 1.000, 0.7933),
    "lowpass_4k":  (10, 1.000, 0.8475),
    "resample_8k": (10, 1.000, 0.7697),
}
print(f"{'attack':12s} | {'AudioSeal det':>13s} {'bits':>6s} {'WER':>6s} | {'WavMark det':>11s} {'bits':>6s} {'WER':>6s} | winner")
print("-" * 95)
wins = {"AudioSeal": 0, "WavMark": 0, "tie": 0}
for name, label, _ in [("clean_wm", "No attack", None)] + ATTACKS:
    a_det, a_bits, a_wer = AUDIOSEAL_REF[name]
    r = pivot[pivot["attack"] == name].iloc[0]
    w_det, w_bits, w_wer = int(r["detected"]), float(r["mean_bits"]), float(r["mean_wer"])
    # Winner is judged by detection first, then bit accuracy; WER is audio quality, not watermark robustness
    if (w_det, w_bits) > (a_det, a_bits):
        winner, wins["WavMark"] = "WavMark", wins["WavMark"] + 1
    elif (w_det, w_bits) < (a_det, a_bits):
        winner, wins["AudioSeal"] = "AudioSeal", wins["AudioSeal"] + 1
    else:
        winner, wins["tie"] = "tie", wins["tie"] + 1
    print(f"{name:12s} | {a_det:>10d}/10 {a_bits:>6.0%} {a_wer:>6.0%} | {w_det:>8d}/10 {w_bits:>6.0%} {w_wer:>6.0%} | {winner}")
print("-" * 95)
print(f"Summary: AudioSeal better in {wins['AudioSeal']} conditions, WavMark better in {wins['WavMark']}, {wins['tie']} tied.")
print()
print("What happened to the speed hypothesis? AudioSeal detected 0/10 under speed, but bits stayed 95–97%;")
print("If WavMark also scores 0/10 under speed but with lower bits, the hypothesis is confirmed —")
print("i.e. speed breaks WavMark more. If WavMark keeps speed intact, the hypothesis is rejected.")


## Cell 20 — Zip the outputs for download
The zip contains 10 watermarked files (WavMark) + 110 attacked files + the summary table + the chart.
It does not include the clean files — you already have them from the stage-1b zip.


In [ ]:
import os, zipfile
with zipfile.ZipFile("persian-watermark-wavmark.zip", "w") as z:
    for fn in sorted(os.listdir("persian_tts_watermarked_wavmark")):
        z.write(os.path.join("persian_tts_watermarked_wavmark", fn))
    for name, _, _ in ATTACKS:
        for fn in sorted(os.listdir(f"persian_wm_attacked_wavmark/{name}")):
            z.write(os.path.join(f"persian_wm_attacked_wavmark/{name}", fn))
    z.write("summary_wavmark.csv")
    if os.path.exists("chart_wavmark.png"):
        z.write("chart_wavmark.png")
print("Created: persian-watermark-wavmark.zip")
print("Size:", round(os.path.getsize("persian-watermark-wavmark.zip") / 1024 / 1024, 1), "MB")
print("Files: 10 watermarked + 110 attacked + summary_wavmark.csv + chart_wavmark.png")


## Done — what do you send back?
**1. Send me the file `summary_wavmark.csv`**
**2. Send me the file `persian-watermark-wavmark.zip` too**

**And answer these questions:**
- **A)** What did WavMark do against speed — was the hypothesis confirmed or rejected?
- **B)** Which method was more robust overall — AudioSeal or WavMark?
- **C)** Listening: did sentence 1 with the WavMark watermark sound different from the AudioSeal version? (Compare the SNR of both from cell 9 with stage 1b)

**If the comparison is complete, the next steps are (not chosen yet):**
- Test harder attacks: cropping, echo, Opus compression
- Or add the WavMark results to the lab, next to AudioSeal
